# Drifting ICNN

In [1]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import random
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Optional

NOTEBOOK_DIR = Path('Drifting-ICNN') if Path('Drifting-ICNN/npf_drift.py').exists() else Path('.')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))


In [2]:
def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)

### Hyperparameters

In [3]:
METHODS = ["kernel", "ot_direct", "npf"]
NUM_ITERS = 3000
BATCH_SIZE = 128
# The report's MNIST protocol is a low-dimensional latent-space experiment.
# Keeping this at 6 makes Sinkhorn and the ICNN/NPF regression much better
# conditioned than the previous 64D version.
LATENT_DIM = 6
NOISE_DIM = 64
LR_GEN = 2e-4
LR_VAE = 1e-3
VAE_KL_WEIGHT = 1e-3
# Train one VAE once, freeze it, and reuse its whitened latent space for every
# method/reg sweep. 400 steps was enough for smoke tests but gave a poorly
# calibrated latent distribution; 2000 is still light on MNIST and far stabler.
VAE_WARMUP_STEPS = 2000
LATENT_STATS_MAX_BATCHES = None  # None = use the whole training set
SEED = 42
seed_everything(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MNIST_ROOT = "./data"
SAVE_PATH = "results/mnist/mnist_drifting_results.png"


NPF_HIDDEN_DIMS = [128, 128, 128]
NPF_OUTER_RANK = 4
NPF_INNER_RANK = 1
NPF_ACTIVATION = "softplus"
NPF_ELU_ALPHA = 1.0
NPF_SOFTPLUS_BETA = 1.0
NPF_INIT_EPS = 1e-2
NPF_INIT_MODE = "identity"
NPF_INIT_BLEND = 0.25


NPF_OUTER_DELTA_INIT = 0.3


INNER_OBJECTIVE = "regression"        # or "semi_dual"
SEMI_DUAL_PHI_STEPS = 1               # inner φ-steps per ψ-step
SEMI_DUAL_CYCLE_WEIGHT = 50.0         # γ in L_cycle (only used by semi_dual)

RESET_INNER_OPTIMIZER = True


INNER_STEPS = 30
NPF_INNER_LR = 1e-2
NPF_ADAM_BETAS = (0.9, 0.999)
NPF_ADAM_EPS = 1e-8
NPF_WEIGHT_DECAY = 0.0
NPF_GRAD_CLIP = 5.0

GEN_HIDDEN_DIMS = [512, 256, 128]
LATENT_REG_WEIGHT = 5e-2

# Sinkhorn regularization is dimensionless because costs are median-normalized.
SINKHORN_REG = 0.1
SINKHORN_REG_GRID = [0.05, 0.1, 0.2, 0.5, 1.0]
SINKHORN_NUM_ITERS = 80
NORMALIZE_OT_COST = True
EVAL_SINKHORN_REG = 0.05  # used only as a *diagnostic* now, not for selection
EVAL_NUM_BATCHES = 4

SLICED_W2_PROJECTIONS = 256


### 1. The VAE (Encoder/Decoder)
We need this to map Images <-> Latent Space. 
The Drifting Model operates ONLY in the latent space.

In [4]:
class MNISTVAE(nn.Module):
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),  # 28 -> 14
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), # 14 -> 7
            nn.SiLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),# 7 -> 4
            nn.SiLU(),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.SiLU(),
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.SiLU(),
            nn.Linear(256, 128 * 4 * 4),
            nn.SiLU(),
        )
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (128, 4, 4)),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=0), # 4 -> 7
            nn.SiLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),                     # 7 -> 14
            nn.SiLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),                      # 14 -> 28
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor):
        h = self.decoder_fc(z)
        return self.decoder(h)

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


class LatentGenerator(nn.Module):
    """
    Asymmetric latent generator in the whitened VAE latent space.

    With hidden_dims=(512, 256, 128), the Linear stack is:
        NOISE_DIM -> 512 -> 256 -> 128 -> LATENT_DIM
    """
    def __init__(self, noise_dim: int = 64, latent_dim: int = 6,
                 hidden_dims: Tuple[int, ...] = (512, 256, 128)):
        super().__init__()
        dims = [noise_dim, *hidden_dims, latent_dim]
        layers = []
        for in_dim, out_dim in zip(dims[:-2], dims[1:-1]):
            layers += [nn.Linear(in_dim, out_dim), nn.SiLU()]
        layers.append(nn.Linear(dims[-2], dims[-1]))
        self.net = nn.Sequential(*layers)
        self._init_output_scale(target_std=1.0)

    def _init_output_scale(self, target_std: float):
        with torch.no_grad():
            device = self.net[0].weight.device
            test_input = torch.randn(2000, self.net[0].in_features, device=device)
            test_output = self.net(test_input)
            current_std = test_output.std().item()
            if current_std > 1e-6:
                scale_factor = target_std / current_std
                self.net[-1].weight.mul_(scale_factor)
                self.net[-1].bias.mul_(scale_factor)

    def forward(self, eps: torch.Tensor):
        return self.net(eps)


### 2. NPF ICNN & Sinkhorn (NPF parameterization of $\psi_\omega$)
The convex potential is now an NPF (Vesseron & Cuturi, 2024) input convex network: PSD outer base + per-layer convex quadratic injections + non-negative cascade. Both the principled LogNormal init and the identity init are applied jointly — they are complementary, not alternatives. All NPF code lives in [`npf_drift.py`](npf_drift.py) and is shared by every notebook in this folder.

In [5]:
# Shared NPF module. We use the log-domain barycentric Sinkhorn target with
# median-cost normalization both for OT-direct and for NPF's inner-loop target;
# direct-domain Sinkhorn underflows easily even in moderately sized latent spaces.
from npf_drift import (
    NPFInputConvexPotential,
    NPFDriftField,
    barycentric_target_log,
    count_parameters,
)


def make_npf_sinkhorn_target_fn(reg: float):
    """Closure over reg for NPFDriftField.sinkhorn_target_fn."""
    return lambda x, y: barycentric_target_log(
        x, y,
        reg=reg,
        num_iters=SINKHORN_NUM_ITERS,
        normalize_cost=NORMALIZE_OT_COST,
    )


In [6]:



def make_npf_drift(dim: int = LATENT_DIM,
                   hidden_dims=None,
                   inner_steps: int = INNER_STEPS,
                   sinkhorn_reg: float = SINKHORN_REG,
                   inner_lr: float = NPF_INNER_LR,
                   init_eps: float = NPF_INIT_EPS,
                   outer_rank: int = NPF_OUTER_RANK,
                   inner_rank: int = NPF_INNER_RANK,
                   activation: str = NPF_ACTIVATION,
                   softplus_beta: float = NPF_SOFTPLUS_BETA,
                   grad_clip: float = NPF_GRAD_CLIP,
                   init_mode: str = NPF_INIT_MODE,
                   init_blend: float = NPF_INIT_BLEND,
                   outer_delta_init: float = NPF_OUTER_DELTA_INIT,
                   inner_objective: str = INNER_OBJECTIVE,
                   semi_dual_phi_steps: int = SEMI_DUAL_PHI_STEPS,
                   cycle_weight: float = SEMI_DUAL_CYCLE_WEIGHT,
                   reset_inner_optimizer: bool = RESET_INNER_OPTIMIZER) -> NPFDriftField:
    """Construct an NPFDriftField with the notebook's tuned MNIST defaults.

    The Sinkhorn target function is wired in for `inner_objective="regression"`
    only; `"semi_dual"` ignores the Sinkhorn target entirely (it learns the
    Brenier map directly via the Makkuva min–max + W2GN cycle term).
    `cycle_weight` is the γ in L_cycle = γ·(‖∇φ(∇ψ(x))−x‖² + ‖∇ψ(∇φ(y))−y‖²),
    which is a no-op for `inner_objective="regression"`.
    """
    if hidden_dims is None:
        hidden_dims = NPF_HIDDEN_DIMS
    return NPFDriftField(
        dim=dim,
        hidden_dims=hidden_dims,
        outer_rank=outer_rank,
        inner_rank=inner_rank,
        activation=activation,
        elu_alpha=NPF_ELU_ALPHA,
        softplus_beta=softplus_beta,
        init_eps=init_eps,
        outer_delta_init=outer_delta_init,
        inner_steps=inner_steps,
        inner_lr=inner_lr,
        adam_betas=NPF_ADAM_BETAS,
        adam_eps=NPF_ADAM_EPS,
        weight_decay=NPF_WEIGHT_DECAY,
        grad_clip=grad_clip,
        sinkhorn_reg=sinkhorn_reg,
        sinkhorn_iters=SINKHORN_NUM_ITERS,
        sinkhorn_target_fn=make_npf_sinkhorn_target_fn(sinkhorn_reg),
        init_mode=init_mode,
        init_blend=init_blend,
        inner_objective=inner_objective,
        semi_dual_phi_steps=semi_dual_phi_steps,
        cycle_weight=cycle_weight,
        reset_inner_optimizer=reset_inner_optimizer,
    )


### 3. Baselines (Kernel & OT Direct) - Adapted for Latent Space

In [7]:
def compute_V_kernel(x_gen: torch.Tensor, y_pos: torch.Tensor,
                     tau_list: Tuple[float, ...] = (0.5, 1.0, 2.0)):
    """
    Compute the drifting field V(x) following the original Drifting Model
    Algorithm 2 / Eq. 11, adapted to whitened latent space.
    """
    N, D = x_gen.shape
    M = y_pos.shape[0]
    x = x_gen.detach()
    y = y_pos.detach()

    targets = torch.cat([x, y], dim=0)  # [N+M, D]
    dist = torch.cdist(x, targets)      # [N, N+M]

    scale = dist.mean().clamp(min=1e-6)
    dist_normed = dist * (np.sqrt(D) / scale)

    diag_mask = torch.zeros(N, N + M, device=x.device)
    diag_mask[:, :N] = torch.eye(N, device=x.device)
    dist_normed = dist_normed + diag_mask * 100.0

    V = torch.zeros_like(x)
    for tau in tau_list:
        logits = -dist_normed / tau
        aff_row = torch.softmax(logits, dim=-1)
        aff_col = torch.softmax(logits, dim=-2)
        affinity = torch.sqrt((aff_row * aff_col).clamp(min=1e-6))

        aff_neg = affinity[:, :N]
        aff_pos = affinity[:, N:]
        sum_pos = aff_pos.sum(-1, keepdim=True)
        sum_neg = aff_neg.sum(-1, keepdim=True)

        r_coeff_neg = -aff_neg * sum_pos
        r_coeff_pos = aff_pos * sum_neg
        R_coeff = torch.cat([r_coeff_neg, r_coeff_pos], dim=1)
        force_R = R_coeff @ targets - R_coeff.sum(-1, keepdim=True) * x
        V = V + force_R

    return V.detach()


def compute_V_ot_direct(x_gen: torch.Tensor, y_pos: torch.Tensor,
                        reg: float = SINKHORN_REG,
                        num_iters: int = SINKHORN_NUM_ITERS,
                        normalize_cost: bool = NORMALIZE_OT_COST):
    """Direct OT displacement V(x) = T_Sinkhorn(x) - x in whitened latent space."""
    x = x_gen.detach()
    y = y_pos.detach()
    y_target = barycentric_target_log(
        x, y,
        reg=reg,
        num_iters=num_iters,
        normalize_cost=normalize_cost,
    )
    return (y_target - x).detach()


### 4. Training Loop for MNIST

In [8]:
def _next_batch(data_iter, loader):
    try:
        x_real, _ = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        x_real, _ = next(data_iter)
    return x_real, data_iter


def build_mnist_loader(batch_size=BATCH_SIZE):
    transform = transforms.Compose([transforms.ToTensor()])
    dataset = datasets.MNIST(root=MNIST_ROOT, train=True, download=True, transform=transform)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )


def train_shared_vae(loader, device=DEVICE):
    device = torch.device(device)
    vae = MNISTVAE(latent_dim=LATENT_DIM).to(device)
    vae_optim = optim.Adam(vae.parameters(), lr=LR_VAE)
    data_iter = iter(loader)

    print(f"[vae] Warming up shared {LATENT_DIM}D VAE for {VAE_WARMUP_STEPS} steps...")
    vae.train()
    for step in range(VAE_WARMUP_STEPS):
        x_real, data_iter = _next_batch(data_iter, loader)
        x_real = x_real.to(device, non_blocking=True)

        recon, mu, logvar = vae(x_real)
        rec_loss = F.binary_cross_entropy(recon, x_real)
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        vae_loss = rec_loss + VAE_KL_WEIGHT * kl_loss

        vae_optim.zero_grad(set_to_none=True)
        vae_loss.backward()
        vae_optim.step()

        if step in {0, VAE_WARMUP_STEPS - 1}:
            print(
                f"[vae] step {step}/{VAE_WARMUP_STEPS} | "
                f"rec={rec_loss.item():.4f} | kl={kl_loss.item():.4f}"
            )

    vae.eval()
    print("[vae] Warmup done; freezing VAE for all methods.")
    for p in vae.parameters():
        p.requires_grad_(False)
    return vae


def estimate_latent_stats(vae, loader, device=DEVICE, max_batches=None):
    """Estimate a full 6D whitening transform for deterministic encoder means."""
    if max_batches is None:
        max_batches = LATENT_STATS_MAX_BATCHES
    device = torch.device(device)
    latents = []
    vae.eval()
    with torch.no_grad():
        for batch_idx, (x_real, _) in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break
            x_real = x_real.to(device, non_blocking=True)
            mu, _ = vae.encode(x_real)
            latents.append(mu.detach())

    z = torch.cat(latents, dim=0)
    mean = z.mean(dim=0, keepdim=True)
    xc = z - mean
    cov = (xc.T @ xc) / max(z.shape[0] - 1, 1)
    cov = cov + 1e-4 * torch.eye(LATENT_DIM, device=device, dtype=z.dtype)
    evals, evecs = torch.linalg.eigh(cov)
    evals = evals.clamp_min(1e-5)
    whitening = evecs @ torch.diag(evals.rsqrt()) @ evecs.T
    unwhitening = evecs @ torch.diag(evals.sqrt()) @ evecs.T

    z_white = (z - mean) @ whitening.T
    stats = {
        "mean": mean,
        "whitening": whitening,
        "unwhitening": unwhitening,
        "raw_second_moment": float(z.pow(2).mean().item()),
        "white_second_moment": float(z_white.pow(2).mean().item()),
    }
    print(
        f"[latent] raw E[z^2]={stats['raw_second_moment']:.4f} | "
        f"whitened E[z^2]={stats['white_second_moment']:.4f}"
    )
    return stats


def whiten_latents(z: torch.Tensor, latent_stats: Dict[str, torch.Tensor]) -> torch.Tensor:
    return (z - latent_stats["mean"]) @ latent_stats["whitening"].T


def unwhiten_latents(z_white: torch.Tensor, latent_stats: Dict[str, torch.Tensor]) -> torch.Tensor:
    return z_white @ latent_stats["unwhitening"].T + latent_stats["mean"]


def prepare_mnist_latent_space(batch_size=BATCH_SIZE, device=DEVICE):
    loader = build_mnist_loader(batch_size=batch_size)
    vae = train_shared_vae(loader, device=device)
    latent_stats = estimate_latent_stats(vae, loader, device=device)
    return {"loader": loader, "vae": vae, "latent_stats": latent_stats}


def latent_moment_regularizer(z: torch.Tensor) -> torch.Tensor:
    """Bug 2 fix: full covariance penalty toward N(0, I).

    The previous version only penalised diagonal variance of z, which the
    generator could satisfy by spreading along each axis individually
    while collapsing onto a low-rank subspace (every off-diagonal entry
    of cov could be ±1 with σ_ii = 1). The matched-target distribution
    is N(0, I), so the right anti-collapse penalty is ‖cov(z) − I‖_F²
    plus a mean-zero term.
    """
    n = z.shape[0]
    zc = z - z.mean(dim=0, keepdim=True)
    cov = (zc.t() @ zc) / max(n - 1, 1)
    eye = torch.eye(cov.shape[0], device=z.device, dtype=z.dtype)
    cov_term = (cov - eye).pow(2).mean()
    mean_term = z.mean(dim=0).pow(2).mean()
    return cov_term + mean_term


def sliced_wasserstein_2(x: torch.Tensor, y: torch.Tensor,
                         n_proj: int = SLICED_W2_PROJECTIONS) -> torch.Tensor:
    """Bug 1 fix: sliced-W2² between equal-mass empirical distributions.

    Random unit projections θ ∈ S^{d-1} reduce W2² to 1D quantile-matching:
        SW2²(μ, ν) = E_θ ∫ |F_μ_θ⁻¹(t) − F_ν_θ⁻¹(t)|² dt
    Estimated by sorting projected samples. Unlike entropic OT, sliced-W2
    is NOT smoothed and therefore NOT minimised by collapsing onto E[y].
    """
    n = min(x.shape[0], y.shape[0])
    x = x[:n]
    y = y[:n]
    d = x.shape[1]
    theta = torch.randn(n_proj, d, device=x.device, dtype=x.dtype)
    theta = theta / theta.norm(dim=1, keepdim=True).clamp_min(1e-12)
    xp = (x @ theta.t()).sort(dim=0).values
    yp = (y @ theta.t()).sort(dim=0).values
    return (xp - yp).pow(2).mean()


def evaluate_latent_match(gen, vae, latent_stats, loader, device=DEVICE,
                          n_batches=EVAL_NUM_BATCHES):
    """Validation diagnostics. Selection metric is `sliced_w2` (Bug 1 fix).

    Returns (and prints) the legacy ``latent_ot_proxy`` (||V_eval||²) for
    backwards compatibility, but DO NOT use it for selection — it is
    minimised by mode collapse onto E[y].
    """
    device = torch.device(device)
    was_training = gen.training
    gen.eval()
    xs, ys = [], []
    data_iter = iter(loader)
    with torch.no_grad():
        for _ in range(n_batches):
            x_real, data_iter = _next_batch(data_iter, loader)
            x_real = x_real.to(device, non_blocking=True)
            mu, _ = vae.encode(x_real)
            y_white = whiten_latents(mu, latent_stats)
            eps = torch.randn(y_white.shape[0], NOISE_DIM, device=device)
            x_white = gen(eps)
            xs.append(x_white.detach())
            ys.append(y_white.detach())

    x_eval = torch.cat(xs, dim=0)
    y_eval = torch.cat(ys, dim=0)
    V_eval = compute_V_ot_direct(
        x_eval, y_eval,
        reg=EVAL_SINKHORN_REG,
        num_iters=SINKHORN_NUM_ITERS,
        normalize_cost=NORMALIZE_OT_COST,
    )
    sw2 = sliced_wasserstein_2(x_eval, y_eval)
    mean_gap = (x_eval.mean(0) - y_eval.mean(0)).pow(2).mean().sqrt().item()
    std_gap = (x_eval.std(0) - y_eval.std(0)).pow(2).mean().sqrt().item()
    metrics = {
        "sliced_w2": float(sw2.item()),
        "latent_ot_proxy": float((V_eval ** 2).mean().item()),
        "mean_gap": mean_gap,
        "std_gap": std_gap,
    }
    if was_training:
        gen.train()
    return metrics


def train_mnist(method='kernel', num_iters=NUM_ITERS, batch_size=BATCH_SIZE,
                lr_gen=LR_GEN, inner_steps=INNER_STEPS, device=DEVICE,
                sinkhorn_reg=SINKHORN_REG, context: Optional[Dict] = None,
                inner_objective: str = INNER_OBJECTIVE):
    device = torch.device(device)
    if context is None:
        context = prepare_mnist_latent_space(batch_size=batch_size, device=device)
    loader = context["loader"]
    vae = context["vae"].to(device)
    latent_stats = context["latent_stats"]
    data_iter = iter(loader)

    gen = LatentGenerator(
        noise_dim=NOISE_DIM,
        latent_dim=LATENT_DIM,
        hidden_dims=tuple(GEN_HIDDEN_DIMS),
    ).to(device)
    gen_optim = optim.Adam(gen.parameters(), lr=lr_gen)

    npf_drift = None
    if method == 'npf':
        npf_drift = make_npf_drift(
            dim=LATENT_DIM,
            hidden_dims=NPF_HIDDEN_DIMS,
            inner_steps=inner_steps,
            sinkhorn_reg=sinkhorn_reg,
            inner_objective=inner_objective,
        ).to(device)
        with torch.no_grad():
            z0 = torch.randn(32, LATENT_DIM, device=device)
            init_err = (npf_drift.psi.gradient(z0, create_graph=False) - z0).pow(2).mean().sqrt().item()
            phi_str = (
                f" | phi_params={count_parameters(npf_drift.phi):,}"
                if npf_drift.phi is not None else ""
            )
            print(
                f"[npf:{inner_objective}] psi_params={count_parameters(npf_drift.psi):,}"
                f"{phi_str} | init RMS |T(z)-z|={init_err:.3e}"
            )
    elif method not in {'kernel', 'ot_direct'}:
        raise ValueError(f"Unknown method: {method}")

    losses, v_norms, snapshots, npf_stats_history = [], [], [], []
    vae.eval()
    snapshot_period = max(1, num_iters // 8)

    for it in range(num_iters):
        x_real_img, data_iter = _next_batch(data_iter, loader)
        x_real_img = x_real_img.to(device, non_blocking=True)

        with torch.no_grad():
            mu, _ = vae.encode(x_real_img)
            y_pos = whiten_latents(mu, latent_stats)

        eps = torch.randn(batch_size, NOISE_DIM, device=device)
        x_gen = gen(eps)

        npf_stats = None
        if method == 'kernel':
            V = compute_V_kernel(x_gen, y_pos, tau_list=(0.5, 1.0, 2.0))
        elif method == 'ot_direct':
            V = compute_V_ot_direct(
                x_gen,
                y_pos,
                reg=sinkhorn_reg,
                num_iters=SINKHORN_NUM_ITERS,
                normalize_cost=NORMALIZE_OT_COST,
            )
        elif method == 'npf':
            V, npf_stats = npf_drift.compute_V_with_stats(x_gen, y_pos)
            npf_stats_history.append(npf_stats)

        target_pts = (x_gen.detach() + V).detach()
        loss_drift = ((x_gen - target_pts) ** 2).mean()
        latent_reg = latent_moment_regularizer(x_gen)
        loss = loss_drift + LATENT_REG_WEIGHT * latent_reg
        v_norm = (V ** 2).mean().item()

        gen_optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        gen_optim.step()

        losses.append(loss.item())
        v_norms.append(v_norm)

        if it % snapshot_period == 0 or it == num_iters - 1:
            with torch.no_grad():
                z_eval_white = gen(torch.randn(64, NOISE_DIM, device=device))
                z_eval_raw = unwhiten_latents(z_eval_white, latent_stats)
                x_eval_img = vae.decode(z_eval_raw).cpu()
                snapshots.append((it, x_eval_img.clone()))
            msg = (
                f"[{method}] Iter {it}/{num_iters} | loss={loss.item():.4f} | "
                f"||V||^2={v_norm:.5f} | reg={sinkhorn_reg}"
            )
            if npf_stats is not None:
                msg += (
                    f" | inner={npf_stats['inner_loss']:.4f}"
                    f" | fit={npf_stats['fit_ratio']:.2f}"
                )
            print(msg)

    metrics = evaluate_latent_match(gen, vae, latent_stats, loader, device=device)
    print(
        f"[{method}] eval sw2={metrics['sliced_w2']:.5f} | "
        f"latent_ot={metrics['latent_ot_proxy']:.5f} | "
        f"mean_gap={metrics['mean_gap']:.4f} | std_gap={metrics['std_gap']:.4f}"
    )

    return {
        'losses': losses,
        'v_norms': v_norms,
        'snapshots': snapshots,
        'method': method,
        'sinkhorn_reg': sinkhorn_reg,
        'metrics': metrics,
        'npf_stats': npf_stats_history,
    }


def run_reg_sweep(method='ot_direct', reg_grid=SINKHORN_REG_GRID,
                  num_iters=500, batch_size=BATCH_SIZE, device=DEVICE,
                  context: Optional[Dict] = None):
    """Small regularization sweep using the shared frozen VAE latent space."""
    if context is None:
        context = prepare_mnist_latent_space(batch_size=batch_size, device=device)
    sweep_results = []
    for reg in reg_grid:
        print(f"\n{'='*60}\n{method} with sinkhorn_reg={reg}\n{'='*60}")
        sweep_results.append(
            train_mnist(
                method=method,
                num_iters=num_iters,
                batch_size=batch_size,
                lr_gen=LR_GEN,
                inner_steps=INNER_STEPS,
                device=device,
                sinkhorn_reg=reg,
                context=context,
            )
        )
    return sweep_results


In [9]:
# Prepare one shared latent space, tune Sinkhorn regularization on a validation
# sliced-W2 score, then train every method against that same frozen VAE.
mnist_context = prepare_mnist_latent_space(batch_size=BATCH_SIZE, device=DEVICE)

reg_sweep_results = run_reg_sweep(
    method="ot_direct",
    reg_grid=SINKHORN_REG_GRID,
    num_iters=500,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    context=mnist_context,
)


def _reg_score(res):
    # Bug 1 fix: select on sliced-W2, not on ||V_eval||². The latter is
    # minimised by collapse onto E[y]; sliced-W2 is not.
    return res["metrics"]["sliced_w2"]


reg_sweep_summary = sorted(reg_sweep_results, key=_reg_score)
print("\nSinkhorn-reg sweep (sorted by validation sliced-W2):")
for r in reg_sweep_summary:
    print(
        f"  reg={r['sinkhorn_reg']:>5}  "
        f"sw2={r['metrics']['sliced_w2']:.5f}  "
        f"latent_ot={r['metrics']['latent_ot_proxy']:.5f}  "
        f"mean_gap={r['metrics']['mean_gap']:.4f}  "
        f"std_gap={r['metrics']['std_gap']:.4f}"
    )

BEST_SINKHORN_REG = reg_sweep_summary[0]["sinkhorn_reg"]
print(f"\nBest sinkhorn_reg (by sliced-W2) = {BEST_SINKHORN_REG}")

results_mnist = []
for m in METHODS:
    print(f"\n{'='*60}\nTraining MNIST: {m}\n{'='*60}")
    results_mnist.append(
        train_mnist(
            method=m,
            num_iters=NUM_ITERS,
            batch_size=BATCH_SIZE,
            lr_gen=LR_GEN,
            inner_steps=INNER_STEPS,
            device=DEVICE,
            sinkhorn_reg=BEST_SINKHORN_REG,
            context=mnist_context,
            inner_objective=INNER_OBJECTIVE,
        )
    )


[vae] Warming up shared 6D VAE for 2000 steps...
[vae] step 0/2000 | rec=0.6911 | kl=0.0013
[vae] step 1999/2000 | rec=0.1112 | kl=3.7674
[vae] Warmup done; freezing VAE for all methods.
[latent] raw E[z^2]=2.6712 | whitened E[z^2]=0.9995

ot_direct with sinkhorn_reg=0.05
[ot_direct] Iter 0/500 | loss=1.6095 | ||V||^2=1.54080 | reg=0.05
[ot_direct] Iter 62/500 | loss=0.1757 | ||V||^2=0.17376 | reg=0.05
[ot_direct] Iter 124/500 | loss=0.1586 | ||V||^2=0.15671 | reg=0.05
[ot_direct] Iter 186/500 | loss=0.1832 | ||V||^2=0.18038 | reg=0.05
[ot_direct] Iter 248/500 | loss=0.2004 | ||V||^2=0.19795 | reg=0.05
[ot_direct] Iter 310/500 | loss=0.1777 | ||V||^2=0.17533 | reg=0.05
[ot_direct] Iter 372/500 | loss=0.1760 | ||V||^2=0.17388 | reg=0.05
[ot_direct] Iter 434/500 | loss=0.1987 | ||V||^2=0.19643 | reg=0.05
[ot_direct] Iter 496/500 | loss=0.2003 | ||V||^2=0.19818 | reg=0.05
[ot_direct] Iter 499/500 | loss=0.1636 | ||V||^2=0.16180 | reg=0.05
[ot_direct] eval sw2=0.07172 | latent_ot=0.10785 |

### Visualization

In [10]:
def plot_mnist_results(results_list, save_path=SAVE_PATH):
    n_methods = len(results_list)
    n_snaps = len(results_list[0]['snapshots'])

    fig, axes = plt.subplots(n_methods, n_snaps, figsize=(3.5 * n_snaps, 3.5 * n_methods))
    if n_methods == 1:
        axes = np.array([axes])
    if n_snaps == 1:
        axes = axes.reshape(n_methods, 1)

    for row, res in enumerate(results_list):
        for col, (it, imgs) in enumerate(res['snapshots']):
            ax = axes[row, col]
            grid = make_grid(imgs[:16], nrow=4, normalize=True, value_range=(0, 1))
            grid_np = grid.permute(1, 2, 0).squeeze(-1).numpy()
            ax.imshow(grid_np, cmap='gray')
            ax.set_title(f"{res['method']} | iter {it}")
            ax.axis('off')

    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved MNIST results to {save_path}")
    plt.show()
    plt.close(fig)

if 'results_mnist' in globals() and len(results_mnist) > 0:
    plot_mnist_results(results_mnist)
else:
    print("Run the training cell first to populate `results_mnist`.")

Saved MNIST results to results/mnist/mnist_drifting_results.png


/tmp/ipykernel_75809/3899198153.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
